# 02 Speech LLM: Whisper-small encoder + Qwen2.5-0.5B (LoRA), ASR + SLU from audio

In [ ]:
# Settings: GPU T4 x2, Internet ON. Persistent outputs: results/ and runs/ in /kaggle/working.
import os, shutil, subprocess
REPO_URL = "https://github.com/ByteBoss123/speaker-aware-speech-llm.git"
WORK = "/kaggle/working/speaker-aware-speech-llm"
if not os.path.exists(WORK):
    src = "/kaggle/input/speaker-aware-speech-llm"   # alternative: upload the repo as a Kaggle dataset
    if os.path.exists(src):
        shutil.copytree(src, WORK)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, WORK], check=True)
os.chdir(WORK)
DATA = "/tmp/data"   # large ephemeral disk; /kaggle/working is capped at ~20 GB
os.environ["DATA"] = DATA

In [ ]:
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /tmp /kaggle/working | tail -2

In [ ]:
!bash scripts/prepare_data.sh $DATA slurp libri_test libri_train100

Whisper-small baseline (2 GPUs in parallel shards)

In [ ]:
!python scripts/asr_whisper.py --data_root $DATA --set librispeech_test_clean --out results/asr

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python scripts/asr_whisper.py --data_root $DATA --set slurp_test --shard 0/2 --out results/asr > /tmp/a0.log 2>&1 & CUDA_VISIBLE_DEVICES=1 python scripts/asr_whisper.py --data_root $DATA --set slurp_test --shard 1/2 --out results/asr > /tmp/a1.log 2>&1; wait
!tail -3 /tmp/a0.log /tmp/a1.log
!python scripts/asr_whisper.py --data_root $DATA --set slurp_test --merge --out results/asr

Train (DDP, 2 GPUs). Raise MAX_STEPS if the throughput printed in the log leaves time in the session.

In [ ]:
MAX_STEPS = 3000
!torchrun --nproc_per_node 2 scripts/sllm_train.py --data_root $DATA --out_dir runs/sllm --max_steps {MAX_STEPS}

In [ ]:
!python scripts/sllm_eval.py --data_root $DATA --run_dir runs/sllm --out results/sllm

GRPO directly on the speech LLM (audio -> JSON), reference = merged SFT model

In [ ]:
!python scripts/sllm_grpo.py --data_root $DATA --sft_dir runs/sllm --out_dir runs/sllm_grpo --steps 200

In [ ]:
!python scripts/sllm_eval.py --data_root $DATA --run_dir runs/sllm --grpo_dir runs/sllm_grpo --out results/sllm_grpo

In [ ]:
!python scripts/summarize.py results runs
!cd /kaggle/working && zip -qr results_sllm.zip speaker-aware-speech-llm/results speaker-aware-speech-llm/runs -x '*.npz'